# Build a MovieLens recommendation baseline with Nori

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/synthefy-nori/blob/codex/movielens-recommendation-notebook/examples/notebooks/nori-movielens-recommendations.ipynb)

Predict movie ratings from user attributes, movie metadata, and rating history with Nori-6M. Compare rating error with a movie-average baseline and the SVD++ collaborative-filtering baseline from the Surprise library, then recommend ten unseen movies to an example user.

This is the runnable companion to [Build a MovieLens Recommendation Baseline with Nori in Minutes](https://synthefy.com/blog/nori-recommendation-systems). Nori uses labeled examples as context; its pretrained weights stay fixed. Candidate selection and sorting happen in the code below.

Use Python 3.11 or 3.12 and run the cells in order. For the full 80,000-context/20,000-query split, use a GPU with at least 40 GB of memory, such as an A100 or H100 (**Colab: Runtime → Change runtime type**). On a smaller GPU or CPU, set `SMOKE_TEST = True` in the configuration cell to try the workflow on 512 context rows and 32 test rows. That subset is a functionality check, not a reproduction of the blog results. The full run keeps every context row and has a larger memory requirement. No API key, private repository, or local dataset is required.

## 1. Install the public packages

The SDK and local runtime versions below were the latest stable releases checked on October 4, 2026. Run this cell before importing packages; restart the kernel if you have already imported different versions. The notebook prints the actual versions used for your run.

In [ ]:
%pip install -q "synthefy==7.1.3" "synthefy-nori==0.21.0" "scikit-surprise==1.1.5" "huggingface-hub>=1,<2"

In [ ]:
import hashlib
import io
import json
import os
import urllib.request
import zipfile
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from surprise import Dataset, Reader, SVDpp
from synthefy_nori import MemoryPolicy, NoriRegressor

DATA_DIR = Path("data/ml-100k")
DEVICE = os.environ.get("MOVIELENS_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
# Set True for a small end-to-end check. These subset scores are NOT the blog benchmark.
SMOKE_TEST = False
SEED = 0
SHRINK = 5.0
GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western",
]
PACKAGE_VERSIONS = {
    name: version(name)
    for name in ("synthefy", "synthefy-nori", "numpy", "pandas", "torch", "scikit-surprise")
}
print(json.dumps({"packages": PACKAGE_VERSIONS, "device": DEVICE, "smoke_test": SMOKE_TEST}, indent=2))

## 2. Download MovieLens 100K and inspect the tables

[MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) contains 100,000 ratings from 943 users for 1,682 movies. We use its supplied `u1.base` and `u1.test` files, containing 80,000 and 20,000 ratings respectively. This is a rating holdout, not a chronological deployment simulation.

The download extracts only the four files we need and checks their SHA-256 hashes. The GroupLens [dataset page](https://grouplens.org/datasets/movielens/100k/) includes the usage terms. Dataset reference: Harper and Konstan, [The MovieLens Datasets: History and Context](https://doi.org/10.1145/2827872), 2015.

In [ ]:
DATA_SHA256 = {
    "u1.base": "ce253ec86c448b44fb3ba9a30d12dcfc2e9210cbde71efada3730c22e9ac212a",
    "u1.test": "18c6014a4b2c7324f250a63f8904a7b16b2b19f911129e346141507b0cbac950",
    "u.user": "f120e114da2e8cf314fd28f99417c94ae9ddf1cb6db8ce0e4b5995d40e90e62c",
    "u.item": "553841ebc7de3a0fd0d6b62a204ea30c1e651aacfb2814c7a6584ac52f2c5701",
}


def download_movielens_files(data_dir):
    """Download the four benchmark files and verify their contents."""
    data_dir.mkdir(parents=True, exist_ok=True)
    if not all((data_dir / name).exists() for name in DATA_SHA256):
        url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        with urllib.request.urlopen(url, timeout=120) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            for name, expected in DATA_SHA256.items():
                payload = archive.read(f"ml-100k/{name}")
                if hashlib.sha256(payload).hexdigest() != expected:
                    raise ValueError(f"Unexpected dataset contents: {name}")
                (data_dir / name).write_bytes(payload)
    for name, expected in DATA_SHA256.items():
        if hashlib.sha256((data_dir / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Cached {name} does not match MovieLens 100K; remove it and rerun.")


def read_movielens_ratings(name):
    """Load one supplied rating split, preserving its row order."""
    return pd.read_csv(
        DATA_DIR / name, sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
    )


download_movielens_files(DATA_DIR)
train_ratings_table = read_movielens_ratings("u1.base")
test_ratings_table = read_movielens_ratings("u1.test")
raw_users = pd.read_csv(
    DATA_DIR / "u.user", sep="|", names=["user_id", "age", "gender", "occupation", "zip"]
)
raw_items = pd.read_csv(DATA_DIR / "u.item", sep="|", encoding="latin-1", header=None)
movie_titles = raw_items[[0, 1]].set_axis(["item_id", "title"], axis=1)

assert len(train_ratings_table) == 80_000 and len(test_ratings_table) == 20_000
assert len(raw_users) == 943 and len(raw_items) == 1_682
if SMOKE_TEST:
    train_ratings_table = train_ratings_table.sample(n=512, random_state=SEED).reset_index(drop=True)
    test_ratings_table = test_ratings_table.sample(n=32, random_state=SEED).reset_index(drop=True)
    print("SMOKE TEST: subset only; do not compare these scores with the blog benchmark.")

users = raw_users.drop(columns="zip").copy()
for column in ("gender", "occupation"):
    categories = pd.Categorical(users[column])
    print(f"{column} codes:", dict(enumerate(categories.categories)))
    users[column] = categories.codes
users["age_band"] = users["age"] // 10
items = raw_items[[0, 2, *range(5, 24)]].set_axis(["item_id", "release_date", *GENRES], axis=1)
items["year"] = pd.to_datetime(items.pop("release_date"), format="%d-%b-%Y").dt.year

print(f"Context ratings: {len(train_ratings_table):,}; held-out ratings: {len(test_ratings_table):,}")
display(train_ratings_table.head())
display(raw_users.drop(columns="zip").head())
display(movie_titles.merge(items[["item_id", "year", "Action", "Animation", "Comedy"]], on="item_id").head())

## 3. Build the 37 input features

Each row describes a user–movie pair. Inputs include user ID, movie ID, age, gender code, occupation code, age band, 19 genre indicators, release year, and rating-history statistics. Movie titles are display labels, not model inputs.

The history features are the movie's mean rating; its mean within gender, age-band, and occupation groups; the user's mean rating; the count behind each mean; and genre affinity. We smooth means toward the available history's global mean with a pseudo-count of five. Genre affinity pools the user's ratings across the target movie's genres; a past movie sharing multiple genres contributes once per matching genre.

**Keep each training row's label out of its own features.** Assign training rows to five folds using `default_rng(1)`. Each row receives statistics from the other four folds. Held-out and candidate rows receive statistics from the full training history. Neither `rating` nor `timestamp` enters the feature matrix. A missing release year stays missing for Nori to handle.

In [ ]:
def join_pair_metadata(pairs):
    """Attach user and movie attributes while preserving pair order."""
    return (
        pairs.merge(users, on="user_id", how="left", validate="many_to_one")
        .merge(items, on="item_id", how="left", validate="many_to_one")
    )


def compute_group_rating_means(history, pairs, keys):
    """Look up smoothed rating means and counts for each pair's group."""
    stats = history.groupby(keys)["rating"].agg(["sum", "count"])
    joined = pairs[keys].join(stats, on=keys).fillna(0)
    mean = (joined["sum"] + SHRINK * history["rating"].mean()) / (joined["count"] + SHRINK)
    return pd.DataFrame({"mean": mean, "n": joined["count"]})


def compute_user_genre_affinity(history, pairs):
    """Pool each user's historical ratings over the target movie's genres."""
    genre_sum = (history[GENRES] * history["rating"].to_numpy()[:, None]).groupby(history["user_id"]).sum()
    genre_count = history.groupby("user_id")[GENRES].sum()
    flags = pairs[GENRES].to_numpy()
    total = np.nan_to_num(genre_sum.reindex(pairs["user_id"]).to_numpy() * flags).sum(axis=1)
    count = np.nan_to_num(genre_count.reindex(pairs["user_id"]).to_numpy() * flags).sum(axis=1)
    return pd.Series((total + SHRINK * history["rating"].mean()) / (count + SHRINK), index=pairs.index)


def build_pair_features(history, pairs):
    """Build numeric pair features from a separate labeled history."""
    features = pairs.drop(columns=["rating", "timestamp"], errors="ignore").copy()
    groups = [
        ("item_mean", ["item_id"]),
        ("item_mean_by_gender", ["item_id", "gender"]),
        ("item_mean_by_age_band", ["item_id", "age_band"]),
        ("item_mean_by_occupation", ["item_id", "occupation"]),
        ("user_mean", ["user_id"]),
    ]
    for name, keys in groups:
        stats = compute_group_rating_means(history, pairs, keys)
        features[name] = stats["mean"]
        features[f"{name}_n"] = stats["n"]
    features["genre_affinity"] = compute_user_genre_affinity(history, pairs)
    return features


def build_context_features(history):
    """Create five-fold history features without each row's own label."""
    folds = np.random.default_rng(1).integers(0, 5, len(history))
    return pd.concat([
        build_pair_features(history.loc[folds != fold], history.loc[folds == fold])
        for fold in range(5)
    ]).loc[history.index]


history = join_pair_metadata(train_ratings_table)
# Only pair IDs are supplied when constructing query features.
test_pairs = join_pair_metadata(test_ratings_table[["user_id", "item_id"]])
train_features = build_context_features(history)
test_features = build_pair_features(history, test_pairs)
train_ratings = train_ratings_table["rating"]
test_ratings = test_ratings_table["rating"]
feature_columns = train_features.columns.tolist()

assert len(feature_columns) == 37
assert test_features.columns.tolist() == feature_columns
assert not {"rating", "timestamp", "title"}.intersection(feature_columns)
print("Input columns:", feature_columns)
shown = ["user_id", "item_id", "age", "gender", "occupation", "year", "Action", "Animation", "Comedy",
         "item_mean", "user_mean", "genre_affinity"]
print("Context examples (12 of 37 input columns, plus the separate target):")
display(train_features[shown].head().assign(rating=train_ratings.head()))
print("Query examples (ratings withheld):")
display(test_features[shown].head())

## 4. Predict ratings with Nori

`fit` supplies the context; it does not update pretrained weights. `allow_subsample=False` retains the full context. The runtime reports which memory path it used. Clip predictions to the one-to-five rating scale for RMSE; leave scores unrounded for ranking.

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
rating_model = NoriRegressor(
    model="nori-6m",
    device=DEVICE,
    encode_id_cols=["user_id", "item_id"],
    memory_policy=MemoryPolicy(cache=True, allow_subsample=False),
)
rating_model.fit(train_features, train_ratings)
predicted_ratings = np.clip(rating_model.predict(test_features), 1, 5)
assert np.isfinite(predicted_ratings).all()
assert rating_model.memory_report_["dropped_context_rows"] == 0
print("Memory path:", rating_model.memory_report_["rung"])
display(test_ratings_table[["user_id", "item_id", "rating"]].head().assign(prediction=predicted_ratings[:5]))

## 5. Measure rating accuracy against the baselines

**SVD++ is an established collaborative-filtering baseline for predicting ratings.** Yehuda Koren of AT&T Labs Research introduced it in his [KDD 2008 paper, *Factorization Meets the Neighborhood*](https://www.cs.cornell.edu/courses/cs6241/2020sp/readings/Koren-2008-Factorization.pdf). It learns user and movie representations from rating history and also uses which movies a user has rated, regardless of the rating values. That makes it a natural reference for MovieLens's sparse user–movie rating table.

We use the implementation in [Surprise](https://joss.theoj.org/papers/10.21105/joss.02174), the open-source Python recommender-system library created by Nicolas Hug. SVD++ has the lowest RMSE among the methods in the library's [published MovieLens 100K benchmark](https://github.com/NicolasHug/Surprise#benchmarks), making it a useful reference for a first baseline. That benchmark averages five cross-validation folds; the results below are our own measurements on the supplied `u1.base` / `u1.test` split. Our run uses its [documented default hyperparameters](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.SVDpp), with `random_state=0` for repeatability and no hyperparameter search.

SVD++ gives us a personalized model trained specifically for rating prediction to compare against Nori's fixed pretrained weights. Alongside the movie-average baseline, it tests whether Nori can provide a useful first recommendation baseline. This comparison does not establish state-of-the-art performance on MovieLens.

Lower root mean squared error (RMSE) is better. The movie-average baseline uses the same smoothed training means as `item_mean`. SVD++ sees only user IDs, movie IDs, and ratings; Nori sees the richer 37-column table. These are comparisons of complete approaches, not identical feature inputs.

The blog's internal results on this split were **Nori-6M 0.929**, **SVD++ (Surprise default settings) 0.933**, and **movie average 1.037** RMSE. The next cell computes your run's results rather than returning those constants. Package versions, hardware, and memory execution paths may affect repeat runs. A single split does not establish statistical significance.

The original evaluation corrected rating-derived features to use the five-fold construction above after inspecting a test result. Treat it as an exploratory benchmark, not an untouched final test. Its rating RMSE does not measure recommendation-list quality or performance on entirely new users.

In [ ]:
def compute_rating_rmse(actual, predicted):
    """Measure squared rating error after clipping to the rating scale."""
    return float(np.sqrt(np.mean((np.asarray(actual) - np.clip(predicted, 1, 5)) ** 2)))


surprise_data = Dataset.load_from_df(
    train_ratings_table[["user_id", "item_id", "rating"]], Reader(rating_scale=(1, 5))
)
svd_model = SVDpp(random_state=SEED).fit(surprise_data.build_full_trainset())
svd_predictions = np.array([
    svd_model.predict(user_id, item_id).est
    for user_id, item_id in test_ratings_table[["user_id", "item_id"]].itertuples(index=False, name=None)
])
results = pd.DataFrame([
    {"method": "Movie average", "RMSE": compute_rating_rmse(test_ratings, test_features["item_mean"])},
    {"method": "SVD++ (Surprise defaults)", "RMSE": compute_rating_rmse(test_ratings, svd_predictions)},
    {"method": "Nori-6M", "RMSE": compute_rating_rmse(test_ratings, predicted_ratings)},
])
print("SMOKE TEST subset results" if SMOKE_TEST else "MovieLens 100K: u1.base → u1.test")
display(results.round({"RMSE": 4}))

## 6. Recommend ten unseen movies to one user

Select an existing user. All catalog movies they have not rated in the context are candidates, including movies whose held-out ratings we must not inspect when scoring. Candidate statistics use only the context history. Break score ties by movie ID and add titles after inference.

This returns recommendations; it does not evaluate their ranking quality. To evaluate top-K lists, fix the candidate set, relevance rule, and K, then measure a ranking metric such as Recall@K or NDCG@K. Evaluate users with no history separately.

In [ ]:
example_user_id = int(train_ratings_table["user_id"].iloc[0])
seen_item_ids = train_ratings_table.loc[
    train_ratings_table["user_id"] == example_user_id, "item_id"
]
unseen_item_ids = items.loc[~items["item_id"].isin(seen_item_ids), "item_id"]
if SMOKE_TEST:
    unseen_item_ids = unseen_item_ids.head(20)
candidate_pairs = pd.DataFrame({"user_id": example_user_id, "item_id": unseen_item_ids.to_numpy()})
candidate_features = build_pair_features(history, join_pair_metadata(candidate_pairs))[feature_columns]
candidates = candidate_pairs.assign(score=rating_model.predict(candidate_features))
assert np.isfinite(candidates["score"]).all()
assert not candidates["item_id"].isin(seen_item_ids).any()
assert not candidates.duplicated(["user_id", "item_id"]).any()
top_10 = (
    candidates.sort_values(["score", "item_id"], ascending=[False, True])
    .head(10)
    .merge(movie_titles, on="item_id", how="left", validate="one_to_one")
    [["user_id", "item_id", "title", "score"]]
)
display(top_10)

## Adapt the baseline to your data

Replace the ratings and metadata tables, preserve the separation between historical labels and query features, and choose a split that matches the decision your product will make. For time-dependent data, compute history features using only information available before each prediction. Keep a simple baseline, evaluate new users separately, and measure the quality of the lists your users will actually see.

See the [Nori quickstart](https://docs.synthefy.com/nori/quickstart) for the prediction interface or [contact Synthefy](https://synthefy.com/contact) to discuss your recommendation task.